# House Price Prediction: Exploratory Data Analysis (EDA)


## Objectives of this EDA:
1. **Dataset Overview**: Understand shapes, data types, and first/last rows.
2. **Missing Value Analysis**: Identify columns with missing entries and formulate imputation strategies.
3. **Duplicate Detection**: Find and drop redundant samples.
4. **Summary Statistics**: Compute central tendency, dispersion, and range metrics.
5. **Distribution of SalePrice**: Check for skewness and analyze log-transformation.
6. **Outlier Detection**: Identify anomalous properties (e.g. very large but cheap homes).
7. **Correlation Heatmap**: Inspect linear relationships between numeric variables and the target.
8. **Pairplots for Core Features**: Visualize interaction distributions of top predictors.
9. **Category Frequency Analysis**: Review categorical balances and cardinalities.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

sys.path.append(str(Path(os.getcwd()).parent))
import src.config as config
import src.utils as utils

## 1. Load the Dataset & Overview

We use the utility function to download the data if it isn't present locally.

In [2]:
utils.download_dataset()

train_df = pd.read_csv(config.TRAIN_DATA_PATH)
test_df = pd.read_csv(config.TEST_DATA_PATH)

print(f"Train dataset dimensions: {train_df.shape}")
print(f"Test dataset dimensions: {test_df.shape}")

train.csv already exists at C:\Users\hp\OneDrive\Desktop\Code\MachineLearning & Regression\house-price-prediction\data\train.csv
test.csv already exists at C:\Users\hp\OneDrive\Desktop\Code\MachineLearning & Regression\house-price-prediction\data\test.csv
Train dataset dimensions: (1460, 81)
Test dataset dimensions: (1459, 80)


In [3]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 2. Data Types and Information

Let's check the schema and types of features.

In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [5]:
dtype_counts = train_df.dtypes.value_counts()
print("Column counts by data type:")
print(dtype_counts)

Column counts by data type:
str        43
int64      35
float64     3
Name: count, dtype: int64


## 3. Duplicate Detection

Check if there are duplicate records (excluding the `Id` column).

In [6]:
feature_cols = [col for col in train_df.columns if col != 'Id']
duplicates = train_df.duplicated(subset=feature_cols).sum()
print(f"Number of duplicate rows in training set: {duplicates}")

Number of duplicate rows in training set: 0


## 4. Missing Value Analysis

We calculate the percentage of missing values per column and plot the top missing features.

In [ ]:
missing_val_counts = train_df.isnull().sum()
missing_percentage = (missing_val_counts / len(train_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_val_counts,
    'Percentage (%)': missing_percentage
}).sort_values(by='Missing Count', ascending=False)

missing_df = missing_df[missing_df['Missing Count'] > 0]
print(f"Total columns with missing values: {len(missing_df)}")
missing_df.head(20)

In [ ]:
# Plot the columns with > 5% missing values
top_missing = missing_df[missing_df['Percentage (%)'] > 5]
if not top_missing.empty:
    sns.barplot(x=top_missing['Percentage (%)'], y=top_missing.index, palette='crest')
    plt.title('Features with > 5% Missing Values')
    plt.xlabel('Percentage of Missing Values (%)')
    plt.ylabel('Features')
    plt.show()

### Missing Values Explanation:
- Features like `PoolQC`, `MiscFeature`, `Alley`, `Fence`, and `FireplaceQu` have high missing percentages because a missing value indicates that the property does not possess that specific amenity (e.g., no pool, no alley access, no fence). In the preprocessing pipeline, we can impute categorical variables with a constant (like 'None') or 'most_frequent'.
- Numerical variables like `LotFrontage` and `GarageYrBlt` have missing values. `GarageYrBlt` is missing when there is no garage. In the pipeline, we impute numerical variables with the median.

## 5. Summary Statistics

Let's get basic statistics of key numerical attributes.

In [ ]:
train_df.describe().T.head(15)

## 6. Target Variable (SalePrice) Distribution

It is critical to inspect the distribution of `SalePrice` since regression models assume normal distributions of residuals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram & KDE
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='blue')
axes[0].set_title(f"Original SalePrice Distribution (Skewness: {train_df['SalePrice'].skew():.2f})")
axes[0].set_xlabel('SalePrice ($)')

# Probability Plot (QQ Plot)
stats.probplot(train_df['SalePrice'], plot=axes[1])
axes[1].set_title("Probability Plot (QQ Plot)")

plt.tight_layout()
plt.show()

### Log-Transform Analysis:
The target variable `SalePrice` exhibits a right-skewed distribution. Log-transforming it can normalize it, which often improves the performance of linear algorithms.

In [ ]:
log_saleprice = np.log1p(train_df['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Log Transformed Histogram & KDE
sns.histplot(log_saleprice, kde=True, ax=axes[0], color='green')
axes[0].set_title(f"Log-Transformed SalePrice (Skewness: {log_saleprice.skew():.2f})")
axes[0].set_xlabel('Log(SalePrice + 1)')

# Log Transformed QQ Plot
stats.probplot(log_saleprice, plot=axes[1])
axes[1].set_title("Log-Transformed QQ Plot")

plt.tight_layout()
plt.show()

The log-transformed target is much closer to a normal distribution, with skewness dropping from 1.88 to 0.12. In our training loop, we will evaluate if standard modeling benefits from a direct target prediction or log target prediction (we stick to predicting direct SalePrice for direct comparison of models as requested, or log-trans internally if needed).

## 7. Outlier Detection

Outliers can significantly distort linear regression coefficients. Let's inspect some key boxplots.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# OverallQual Boxplot
sns.boxplot(x='OverallQual', y='SalePrice', data=train_df, ax=axes[0,0], palette='GnBu')
axes[0,0].set_title('SalePrice vs Overall Quality')

# GrLivArea Scatter with regression line
sns.scatterplot(x='GrLivArea', y='SalePrice', data=train_df, ax=axes[0,1], color='coral', alpha=0.6)
axes[0,1].set_title('SalePrice vs Above Grade Living Area (GrLivArea)')

# GarageCars Boxplot
sns.boxplot(x='GarageCars', y='SalePrice', data=train_df, ax=axes[1,0], palette='Oranges')
axes[1,0].set_title('SalePrice vs Garage Cars Capacity')

# TotalBsmtSF Scatter
sns.scatterplot(x='TotalBsmtSF', y='SalePrice', data=train_df, ax=axes[1,1], color='purple', alpha=0.6)
axes[1,1].set_title('SalePrice vs Total Basement Area (TotalBsmtSF)')

plt.tight_layout()
plt.show()

### Outliers Observation:
- In the `GrLivArea` scatter plot, there are two extreme outliers on the bottom right (GrLivArea > 4000 sq ft, but SalePrice < $300,000). The dataset author recommends removing these as they do not represent typical sales patterns.
- Overall Quality (`OverallQual`) has a clear positive monotonic relationship with price. There are some outlier prices in quality levels 7, 8, and 10.

## 8. Correlation Heatmap

Let's see the linear correlation of top features with SalePrice.

In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns
correlations = train_df[num_cols].corr()['SalePrice'].sort_values(ascending=False)

print("Top 15 positively correlated numerical features:")
print(correlations.head(15))

print("\nTop 5 negatively correlated numerical features:")
print(correlations.tail(5))

In [ ]:
# Top 15 correlated variables matrix plot
top_corr_features = correlations.index[:15]
plt.figure(figsize=(12, 10))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Matrix of Top 15 Features')
plt.show()

### Correlation Observation:
- `OverallQual` (0.79) and `GrLivArea` (0.71) show the highest positive linear correlation to `SalePrice`.
- Garage capacity (`GarageCars`: 0.64 and `GarageArea`: 0.62) and Basement size (`TotalBsmtSF`: 0.61 and `1stFlrSF`: 0.61) also display strong positive correlations.
- Multicollinearity exists between some variables: `GarageCars` & `GarageArea` (0.88), `TotalBsmtSF` & `1stFlrSF` (0.82), and `YearBuilt` & `GarageYrBlt` (0.83). We must use regularization (Lasso/Ridge) or feature selection to handle these.

## 9. Pairplot of Core Predictors

We construct a pairplot for the subset of top features to analyze joint distributions.

In [ ]:
core_features = ['SalePrice', 'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'YearBuilt']
sns.pairplot(train_df[core_features], diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle('Pairwise Relationship Plot of Core Features', y=1.02)
plt.show()

## 10. Category Frequency Analysis

Let's look at key categorical features to examine label distributions and cardinality.

In [ ]:
cat_cols = train_df.select_dtypes(include=['object']).columns
print(f"Total categorical features: {len(cat_cols)}")

# Print unique values for a few key categorical columns
for col in ['MSZoning', 'Neighborhood', 'HouseStyle', 'Foundation']:
    print(f"\nCategory counts for {col} (Unique values: {train_df[col].nunique()}):")
    print(train_df[col].value_counts(dropna=False))

In [ ]:
# Plot distribution for MSZoning and Neighborhood
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(data=train_df, x='MSZoning', ax=axes[0], palette='Set2', hue='MSZoning', legend=False)
axes[0].set_title('Frequency of MSZoning Classifications')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=train_df, y='Neighborhood', ax=axes[1], order=train_df['Neighborhood'].value_counts().index, palette='viridis', hue='Neighborhood', legend=False)
axes[1].set_title('Frequency of Neighborhoods')

plt.tight_layout()
plt.show()

### Categorical Frequency Analysis Explanation:
- `MSZoning` is heavily skewed toward Residential Low Density (RL), showing imbalance.
- `Neighborhood` has high cardinality (25 classes). Some neighborhoods have very few records (like Blueste: 2, NPkVill: 9), whereas others are very frequent (like NAmes: 225, CollgCr: 150). One-hot encoding these will create many sparse columns. Preprocessing with `OneHotEncoder(handle_unknown='ignore')` in Scikit-learn will safely manage unseen categories during inference.